In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score
import pickle

print("Starting Bulletproof Data Preparation...")

# 1. Load Data
df = pd.read_csv('Used Car Dataset.csv')

# 2. Fix Leakage: Drop the ID column and drop all duplicate rows
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
df = df.drop_duplicates()

# 3. Fix the Shifted Rows
valid_owners = ['First Owner', 'Second Owner', 'Third Owner', 'Fourth Owner', 'Fifth Owner']
df = df[df['transmission'].isin(['Manual', 'Automatic'])]
df = df[df['ownsership'].isin(valid_owners)]

# 4. Clean Numerical Columns
# Notice we are ignoring 'max_power(bhp)' because it is corrupted/identical to engine
columns_to_convert = ['mileage(kmpl)', 'engine(cc)']
for col in columns_to_convert:
    if df[col].dtype == 'object':
        df[col] = df[col].str.replace(',', '')
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 5. Advanced Categorical Extraction
# Split "2017 Mercedes-Benz S-Class S400" 
df['brand'] = df['car_name'].str.split(' ', n=2).str[1]      # Gets "Mercedes-Benz"
df['model_name'] = df['car_name'].str.split(' ', n=2).str[2] # Gets "S-Class S400"

# Drop any rows that couldn't be parsed or have missing values
df = df.dropna()

# --- The Complete Features List ---
features = [
    'brand', 
    'model_name',
    'manufacturing_year', 
    'kms_driven', 
    'fuel_type', 
    'transmission', 
    'ownsership', 
    'seats',
    'mileage(kmpl)',
    'engine(cc)', 
    'price(in lakhs)'
]
df = df[features]

# 6. Categorical Encoding
# This will create a high-dimensional dataset (lots of columns) because of all the unique car models
df_encoded = pd.get_dummies(
    df, 
    columns=['brand', 'model_name', 'fuel_type', 'transmission', 'ownsership'], 
    drop_first=True
)

# 7. Split and Train
X = df_encoded.drop('price(in lakhs)', axis=1)
y = df_encoded['price(in lakhs)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {X_train.shape[1]} unique features... (This may take a moment longer)")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 8. Evaluate
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- True Model Performance ---")
print(f"Average Error: {mae:.2f} Lakhs") 
print(f"Model Accuracy Score (R2): {r2*100:.2f}%")

# 9. Save
model_data = {
    'model': model,
    'columns': list(X.columns)
}

with open('car_model.pkl', 'wb') as file:
    pickle.dump(model_data, file)

print("\nSuccess! Uncompromised model saved.")

Starting Bulletproof Data Preparation...
Training on 715 unique features... (This may take a moment longer)

--- True Model Performance ---
Average Error: 3.59 Lakhs
Model Accuracy Score (R2): 70.94%

Success! Uncompromised model saved.


In [32]:
#2017 Honda WR-V i-VTEC S,Jun-17,Comprehensive,Petrol,5,49000,First Owner,Manual,2017,17.5,1199,1199,887,5.85

# Create a dictionary filled with 0s for every single column the model knows about
input_data = {col: 0 for col in X.columns}

# Fill in the numerical details of the Kia
input_data['manufacturing_year'] = "2017"
input_data['kms_driven'] = 49000
input_data['seats'] = 5
input_data['mileage(kmpl)'] = 17.5
input_data['engine(cc)'] = 1199

# Switch the specific category columns from 0 to 1 (True)
if 'brand_Honda' in X.columns: 
    input_data['brand_Honda'] = 1
if 'model_name_WR-V i-VTEC S' in X.columns: 
    input_data['model_name_WR-V i-VTEC S'] = 1
if 'fuel_type_Petrol' in X.columns: 
    input_data['fuel_type_Petrol'] = 1
if 'transmission_Manual' in X.columns: 
    input_data['transmission_Manual'] = 1
if 'ownsership_First Owner' in X.columns: 
    input_data['ownsership_First Owner'] = 1

# Convert our single car to a DataFrame
input_df = pd.DataFrame([input_data])

# Make the prediction
prediction = model.predict(input_df)[0]

print("\n" + "="*40)
print(f"ACTUAL PRICE IN DATASET: 5.85 Lakhs")
print(f"MODEL PREDICTED PRICE:   {prediction:.2f} Lakhs")
print("="*40)


ACTUAL PRICE IN DATASET: 5.85 Lakhs
MODEL PREDICTED PRICE:   6.08 Lakhs
